In [ ]:
# 再実行時: 残った ngrok を終了（トンネル重複・URL取り違い防止）
import time

try:
    from pyngrok import ngrok

    ngrok.kill()
    time.sleep(2)
except Exception:
    pass

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

%cd /content
!rm -rf aibo_v8
!git clone -b colab-stable https://github.com/miya390831-a11y/aibo_v8.git
%cd /content/aibo_v8

# 全パッケージインストール（os.kill なし）
!pip install -q --upgrade nunchaku
!pip install -q --upgrade --force-reinstall "diffusers>=0.36"
!pip install -q "transformers>=4.54" "accelerate>=1.9" "peft>=0.17"
!pip install -q "huggingface_hub>=0.34" "scipy>=1.14"
!pip install -q pyngrok

# importlib をリロード（os.kill 不要）
import importlib, site

importlib.reload(site)

print('✅ Setup 完了')

In [ ]:
%cd /content/aibo_v8
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGINGFACE_HUB_TOKEN'] = userdata.get('HF_TOKEN')

import importlib.util, sys

sys.path.insert(0, '/content/aibo_v8')


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


cfg_mod = load_module('01_config', '01_config.py')
setup_mod = load_module('02_colab_setup', '02_colab_setup.py')
sys_cfg = cfg_mod.SystemConfig()
setup_mod.ColabBootstrap(sys_cfg).run()
print('✅ Bootstrap 完了')

In [ ]:
%cd /content/aibo_v8
main_mod = load_module('07_main', '07_main.py')

m = main_mod.get_instance()
if m is None:
    m = main_mod.AiboMain()
    main_mod._AIBO_MAIN_INSTANCE = m

# A方式: Phase A–F のみ（Gradio は起動しない · Cell4 で FastAPI + ngrok）
if m.orchestrator is None:
    if not m.phase_a_bootstrap():
        raise RuntimeError('Phase A failed')
    if not m.phase_b_resolve_strategy():
        raise RuntimeError('Phase B failed')
    if not m.phase_c_build_pipeline():
        raise RuntimeError('Phase C failed')
    if not m.phase_d_identity():
        print('⚠️ Phase D 部分失敗 · Identity 機能制限の可能性')
    if not m.phase_e_assets():
        raise RuntimeError('Phase E failed')
    if not m.phase_f_orchestrator():
        raise RuntimeError('Phase F failed')
    print('✅ Phase A–F 完了 (orchestrator 構築済み)')
else:
    print('ℹ️ orchestrator 既存 · Phase A–F skip')

print('✅ Main 起動完了 (A方式 · Gradio なし)')

In [ ]:
%cd /content/aibo_v8
import json
import os
import time
import urllib.request
from pyngrok import ngrok
from google.colab import userdata

main_mod = sys.modules.get('07_main') or load_module('07_main', '07_main.py')
m = main_mod.get_instance()
if m is None or m.orchestrator is None:
    raise RuntimeError('Cell3 を先に実行してください (orchestrator 未構築)')

# 同プロセス: attach 済み FastAPI を threading 起動（subprocess 禁止）
os.environ['AIBO_SKIP_BACKGROUND_ORCHESTRATOR'] = '1'
main_mod.launch_fastapi_in_background(
    m.orchestrator,
    m.pipeline_mgr,
    port=8000,
)
time.sleep(10)

# attach 確認
with urllib.request.urlopen('http://127.0.0.1:8000/api/system/status', timeout=30) as resp:
    status = json.loads(resp.read().decode())
if not status.get('orchestrator_attached'):
    raise RuntimeError(f'orchestrator attach 失敗: {status}')
print('✅ FastAPI 起動 · orchestrator_attached: true')

# ngrok 認証 & 公開（1 トンネル · UI は PC ローカル）
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
api_tunnel = ngrok.connect(8000)
api_url = api_tunnel.public_url
print(f'🔌 API URL: {api_url}')
print('')
print('👉 PC の frontend/.env.local にコピー:')
print(f'NEXT_PUBLIC_API_URL={api_url}')